# Flow Length (MFD)

Flow length measures the distance water travels along its flow path. The MFD (Multiple Flow Direction) variant computes proportion-weighted path lengths using MFD fraction grids, where each cell distributes flow to all downslope neighbors.

Two modes are supported:
- **Downstream**: Expected (weighted-average) distance from each cell to its outlet
- **Upstream**: Longest flow path from any drainage divide to each cell

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

from xrspatial import flow_direction_mfd, flow_length_mfd, flow_accumulation_mfd
from xrspatial.terrain import generate_terrain

## Generate terrain

In [ ]:
W = 400
H = 400
terrain = generate_terrain(
    x_range=(-2, 2), y_range=(-2, 2), width=W, height=H, seed=42
)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(terrain.values, cmap='terrain', origin='lower')
ax.set_title('Elevation')
plt.colorbar(im, ax=ax, label='meters')
plt.tight_layout()
plt.show()

## Compute MFD flow direction and accumulation

In [ ]:
mfd_dir = flow_direction_mfd(terrain)
mfd_acc = flow_accumulation_mfd(mfd_dir)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(np.log1p(mfd_acc.values), cmap='Blues', origin='lower')
ax.set_title('MFD flow accumulation (log scale)')
plt.colorbar(im, ax=ax, label='log(1 + accumulation)')
plt.tight_layout()
plt.show()

## Downstream flow length

Downstream flow length gives the proportion-weighted average distance from each cell to the outlet it drains to. Cells near outlets have short distances; cells far upstream have long distances.

In [ ]:
downstream = flow_length_mfd(mfd_dir, direction='downstream')

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(downstream.values, cmap='viridis', origin='lower')
ax.set_title('MFD downstream flow length')
plt.colorbar(im, ax=ax, label='distance (coordinate units)')
plt.tight_layout()
plt.show()

## Upstream flow length

Upstream flow length gives the longest flow path distance from any drainage divide to each cell. Cells on divides have zero length; cells at outlets accumulate the full watershed length.

In [ ]:
upstream = flow_length_mfd(mfd_dir, direction='upstream')

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(upstream.values, cmap='magma', origin='lower')
ax.set_title('MFD upstream flow length')
plt.colorbar(im, ax=ax, label='distance (coordinate units)')
plt.tight_layout()
plt.show()

## Comparison: D8 vs MFD flow length

MFD distributes flow across multiple neighbors, producing smoother flow length fields than D8 which routes everything through the single steepest neighbor.

In [ ]:
from xrspatial import flow_direction, flow_length

d8_dir = flow_direction(terrain)
d8_downstream = flow_length(d8_dir, direction='downstream')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = axes[0].imshow(d8_downstream.values, cmap='viridis', origin='lower')
axes[0].set_title('D8 downstream flow length')
plt.colorbar(im0, ax=axes[0], label='distance')

im1 = axes[1].imshow(downstream.values, cmap='viridis', origin='lower')
axes[1].set_title('MFD downstream flow length')
plt.colorbar(im1, ax=axes[1], label='distance')

plt.tight_layout()
plt.show()